# Truchet Studio

A notebook only about **Truchet tiles**: simple square tiles whose curves always join up, so that a random choice per tile produces long winding paths.

The notebook follows the path from the basic idea to rich results, with the middle steps visible:

| # | Step | What you see |
|---|------|--------------|
| 1 | The principle | Two tiles, and a random grid where the curves always connect |
| 2 | The smoothness dial | Independent coin flips versus smooth fields, side by side |
| 3 | Adding color | Pattern field, a separate color field, and the combination |
| 4 | The constraint, and many more tiles | Why tiles connect, and a library of 14 tiles written as text |
| 5 | The final weave | Choose a tile by index, see the randomness first, then the artwork |

Run the cells from top to bottom.

By **Iman Mossavat**, Fontys ICT.


## 0. Setup and helpers

Imports, small helper functions, and the output function. Run this cell first.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter

from math_vis import (
    make_cmap, smooth_field, squash, finish, fig_to_img, new_rng,
    TILE_TEXT, parse_tile, rotate_tile, tile_curve, P_TILE, N_TILE,
)

# Local helpers specific to this notebook (classic two-arc tile, offset strands, indexing).
def arc(cx, cy, r, deg0, deg1, n=24):
    a = np.radians(np.linspace(deg0, deg1, n))
    return np.column_stack([cx + r * np.cos(a), cy + r * np.sin(a)])

def tile_arcs(i, j, orientation):
    if orientation == 0:
        return [(i, j, 0, 90), (i + 1, j + 1, 180, 270)]
    return [(i + 1, j, 90, 180), (i, j + 1, 270, 360)]

def offset(c, d):
    tang = np.gradient(c, axis=0)
    tang /= np.linalg.norm(tang, axis=1, keepdims=True)
    return c + d * np.column_stack([-tang[:, 1], tang[:, 0]])

def to_index(field, n):
    if n == 1:
        return np.zeros(field.shape, dtype=int)
    cuts = np.quantile(field, np.linspace(0, 1, n + 1)[1:-1])
    return np.digitize(field, cuts)


---
## 1. The principle

**Principle.** Take a square tile with two quarter-circle arcs. The arcs can sit in two orientations, and **either orientation joins up seamlessly with its neighbours**. Flip a coin per tile and the arcs automatically form long winding paths and loops. The order comes from the tile design, and the chaos comes from the coin flips.

**Why it works.** Every arc ends at the *midpoint* of a tile edge, and neighbouring tiles share those midpoints. The red dots below mark them. Whatever orientation each tile gets, the arcs meet.

**Recipe**
1. Draw the two tile types (left).
2. Flip a coin for every tile of a grid. This is the only randomness (independent randomness).
3. Draw each tile in the orientation its coin says.

In [ ]:
SEED = None            # None = new result every run; a number = repeatable
NX, NY = 12, 8         # tiles across and down
rng = new_rng(SEED)

def draw_frame(ax, i, j):
    ax.plot([i, i + 1, i + 1, i, i], [j, j, j + 1, j + 1, j], color="#bbbbbb", lw=0.8, ls="--")
    # red dots at the edge midpoints: every arc ends on one, so neighbours always meet there
    ax.scatter([i + .5, i + 1, i + .5, i], [j, j + .5, j + 1, j + .5], s=18, color="#d1495b", zorder=3)

def draw_tile(ax, i, j, orientation, lw=4, color="#1d3557"):
    for cx, cy, a0, a1 in tile_arcs(i, j, orientation):
        c = arc(cx, cy, 0.5, a0, a1, 40)
        ax.plot(c[:, 0], c[:, 1], color=color, lw=lw, solid_capstyle="butt")

fig = plt.figure(figsize=(12, 6.5))
gs = GridSpec(2, 2, width_ratios=[1, 2.4], figure=fig)

# left: the two possible tiles
for row, (orient, title) in enumerate([(0, "Orientation 0"), (1, "Orientation 1")]):
    ax = fig.add_subplot(gs[row, 0])
    draw_frame(ax, 0, 0); draw_tile(ax, 0, 0, orient)
    ax.set_title(title); ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)

# right: flip a coin per tile, paths appear on their own
ax = fig.add_subplot(gs[:, 1])
coins = rng.integers(0, 2, size=(NY, NX))        # the ONLY randomness: 0 or 1 per tile
for j in range(NY):
    for i in range(NX):
        draw_frame(ax, i, j); draw_tile(ax, i, j, coins[j, i], lw=5)
ax.set_title("Random coin flip per tile → arcs always connect into winding paths")
ax.set_aspect("equal"); ax.axis("off")

fig.text(0.5, 0.02, "Red dots = tile edge midpoints. Every arc ends on one, so any two neighbouring tiles join up, whichever way each was flipped.",
         ha="center", fontsize=10, color="#444")
plt.tight_layout(rect=[0, 0.05, 1, 1]); plt.show()

---
## 2. The smoothness dial

**Principle.** Independent coin flips give a busy, uniform texture. If instead the coins come from a **smooth random field** (blurred noise), neighbouring tiles tend to agree, and the paths organize into broad sweeping regions. The blur radius `sigma` is *the* dial of this piece.

**Recipe**
1. Make one grid of white noise.
2. Blur it by different amounts (`sigma = 0`, `1.2`, `3`). Because it is the **same** noise each time, the comparison is fair.
3. Standardize, then threshold at 0: positive means orientation 1, negative means orientation 0.
4. Draw the tiles.

Top row: the field. Bottom row: the pattern it produces. Notice that the *boundaries between regions* are where the field crosses zero.

In [ ]:
SEED = None
NX, NY = 24, 16
SIGMAS = [0, 1.2, 3.0]          # blur radii to compare (0 = pure independent coin flips)

rng = new_rng(SEED)
noise = rng.normal(size=(NY, NX))                      # one noise grid shared by all three

fig, axs = plt.subplots(2, 3, figsize=(15, 8.6), gridspec_kw={"hspace": 0.28})
for k, sg in enumerate(SIGMAS):
    f = gaussian_filter(noise, sg, mode="wrap") if sg > 0 else noise.copy()
    f = (f - f.mean()) / f.std()
    orientation = (f > 0).astype(int)

    axs[0, k].imshow(f, origin="lower", cmap="RdBu_r", vmin=-2.5, vmax=2.5, interpolation="nearest")
    axs[0, k].set_title(f"sigma = {sg}: the field\n(red > 0 → orientation 1, blue < 0 → orientation 0)")
    axs[0, k].axis("off")

    segs, cols = [], []
    for j in range(NY):
        for i in range(NX):
            for cx, cy, a0, a1 in tile_arcs(i, j, orientation[j, i]):
                segs.append(arc(cx, cy, 0.5, a0, a1))
                cols.append("#c8553d" if orientation[j, i] == 1 else "#1d3557")   # color shows the orientation
    axs[1, k].add_collection(LineCollection(segs, colors=cols, linewidths=2.2))
    axs[1, k].set_xlim(0, NX); axs[1, k].set_ylim(0, NY); axs[1, k].set_aspect("equal"); axs[1, k].axis("off")
    axs[1, k].set_title("the resulting pattern\n(coral = orientation 1, navy = orientation 0)")
plt.tight_layout(); plt.show()

---
## 3. Adding color

**Principle.** Use a **second, independent smooth field** for color, so the color regions do not follow the pattern regions. Two layers of structure interfere, which looks richer than one. Each arc is also drawn as three parallel **strands**, each shifted slightly along the palette, which gives a ribbon with a gradient across its width.

**Recipe**
1. Two independent smooth fields per tile: one for the pattern, one for the color.
2. **Color recipe:** `t = squash(color field) + small jitter`, then `color = palette(t)`.
3. Draw each arc three times at radii `0.5 − BAND`, `0.5`, `0.5 + BAND`. The radii are symmetric about 0.5, which keeps the ribbons continuous across tile edges.
4. Shift each strand along the palette by `STRAND_SHIFT`.

The three panels show the steps; the strip at the bottom is the palette.

In [ ]:
SEED          = None
NX, NY        = 12, 9
ORIENT_SMOOTH = 1.2      # smoothness of the PATTERN field
COLOR_SMOOTH  = 2.5      # smoothness of the COLOR field
BAND          = 0.16     # spacing between the 3 strands
STRAND_SHIFT  = 0.09     # how far each strand drifts along the palette
JITTER        = 0.04     # small independent color variation per arc
BG            = "#f1e9d2"

rng = new_rng(SEED)
palette = make_cmap(["#d9674a", "#e8a56b", "#e6d28f", "#9db08c", "#5f8a86", "#2c3a35"])

orient_field = smooth_field((NY, NX), ORIENT_SMOOTH, rng)   # decides the pattern
color_field  = smooth_field((NY, NX), COLOR_SMOOTH, rng)    # decides the colors
orientation  = (orient_field > 0).astype(int)

def draw_weave(ax, colored, strands=(-1, 0, 1), lw=3.2):
    segs, cols = [], []
    for j in range(NY):
        for i in range(NX):
            for cx, cy, a0, a1 in tile_arcs(i, j, orientation[j, i]):
                # THE COLOR STEP: field value -> 0..1 -> palette, plus a little jitter
                t = squash(color_field[j, i]) + rng.normal(0, JITTER)
                for off in strands:
                    segs.append(arc(cx, cy, 0.5 + off * BAND, a0, a1))
                    cols.append(palette(np.clip(t + off * STRAND_SHIFT, 0, 1)) if colored else "#1d3557")
    ax.add_collection(LineCollection(segs, colors=cols, linewidths=lw, capstyle="butt"))
    ax.set_xlim(0, NX); ax.set_ylim(0, NY); ax.set_aspect("equal"); ax.axis("off")

fig = plt.figure(figsize=(15, 6.8), facecolor=BG)
gs = GridSpec(2, 3, height_ratios=[12, 1], hspace=0.12, figure=fig)

ax1 = fig.add_subplot(gs[0, 0]); ax1.set_facecolor(BG)
draw_weave(ax1, colored=False, strands=(0,), lw=4)
ax1.set_title("1. Pattern field → orientation\n(structure only, one color)")

ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(squash(color_field), cmap=palette, vmin=0, vmax=1, origin="lower",
           extent=(0, NX, 0, NY), interpolation="nearest")
ax2.set_title("2. A SECOND smooth field → color per tile\n(squash to 0..1, look up in palette)")
ax2.set_aspect("equal"); ax2.axis("off")

ax3 = fig.add_subplot(gs[0, 2]); ax3.set_facecolor(BG)
draw_weave(ax3, colored=True)
ax3.set_title("3. Combine: each arc takes its tile's color\n+ jitter, 3 strands shifted along the palette")

axp = fig.add_subplot(gs[1, :])
axp.imshow(np.linspace(0, 1, 512)[None, :], cmap=palette, aspect="auto")
axp.set_yticks([]); axp.set_xticks([0, 511]); axp.set_xticklabels(["t = 0", "t = 1"])
axp.set_title("palette(t)", fontsize=9, pad=2)
for s in axp.spines.values(): s.set_visible(False)
plt.show()

**Things to try.** `COLOR_SMOOTH = 0.01` gives unrelated colors per tile. `JITTER = 0` makes tile borders look blocky. `STRAND_SHIFT = 0` makes flat ribbons. Using the same field for both (`color_field = orient_field`) makes the colors line up with the pattern, which looks more rigid.

---
## 4. The constraint, and many more tiles

**The constraint.** Two neighbouring tiles share an edge, and whatever crosses it must continue into the other tile. If tiles can be rotated, then every edge has to fit against every other edge. That means:

1. **All four edges carry the same pattern.** Rotating by 90° moves the west edge to the north, so they cannot differ.
2. **The pattern must be palindromic**, the same read forwards and backwards. Your neighbour reads the shared edge in the opposite direction.

(Mathematically: edge matching under the rotation group C4, or the dihedral group D4 if flips are allowed. This is the idea behind Wang tiles.)

**The interior is completely free.** Once the edge points are fixed, *any* way of pairing them up is a valid tile.

**Recipe for a tile library**
1. Pick an edge pattern: here **two points per edge**, at 0.3 and 0.7 (mirror images, so palindromic). That gives 8 points, numbered 0 to 7 counter-clockwise from the bottom-left.
2. List every way of pairing the 8 points **without crossing**. That is the 4th Catalan number: 14 tiles.
3. Write each tile as **text**: `"0-5 1-2 3-4 6-7"` means "join point 0 to 5, 1 to 2, ...".
4. Draw each pair as a smooth cubic Bézier that leaves the edge along its normal, so curves continue smoothly across tiles.
5. A rotated tile is the same pairs with every label shifted by 2.

```
          5   4
        +--o---o--+
      6 o         o 3
      7 o         o 2
        +--o---o--+
          0   1
```

**To make even more tiles:** use three points per edge (0.2, 0.5, 0.8), allow crossings (over/under tiles), or fill regions instead of drawing lines.

In [ ]:
# ---------- THE TILE LIBRARY (as text) ----------
# TILE_TEXT is imported from math_vis. These 14 strings are all non-crossing
# pairings of the 8 edge points, but many are just rotations of each other.

# Compute the 6 rotation-equivalence classes and keep one representative each.
def canonical(tile):
    return tuple(sorted(tuple(sorted((i, j))) for i, j in tile))

def rotation_orbit(tile):
    seen = {canonical(tile)}
    for r in range(1, 4):
        seen.add(canonical(rotate_tile(tile, r)))
    return seen

tiles_all = [parse_tile(t) for t in TILE_TEXT]
classes = []
used = set()
for idx, tile in enumerate(tiles_all):
    if idx in used:
        continue
    orbit = rotation_orbit(tile)
    cls = [j for j, other in enumerate(tiles_all) if canonical(other) in orbit]
    for j in cls:
        used.add(j)
    classes.append(cls)
REPS = [cls[0] for cls in classes]
print('Rotation classes:', classes)
print('Representatives :', REPS)

# ---------- SHOW THE REPRESENTATIVES, WITH INDEX NUMBERS AND POINT LABELS ----------
fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
for ax, k in zip(axes.ravel(), REPS):
    ax.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], color='#ccc', lw=0.7, ls='--')
    for i, j in tiles_all[k]:
        c = tile_curve(i, j)
        ax.plot(c[:, 0], c[:, 1], color='#1d3557', lw=2)
    for n_, (x, y) in enumerate(P_TILE):
        nx, ny = N_TILE[n_]
        ax.text(x + 0.12 * nx, y + 0.12 * ny, str(n_),
                fontsize=6, color='#d1495b', ha='center', va='center')
    ax.set_title(f'TILE_ID = {k}  (class {classes[REPS.index(k)]})', fontsize=8)
    ax.set_aspect('equal')
    ax.axis('off')
plt.suptitle('The 6 rotationally distinct non-crossing pairings', fontsize=12)
plt.tight_layout()
plt.show()


### A random grid of the whole library

Every cell gets a **random tile** and a **random rotation**. The curves still join smoothly everywhere, because all 14 tiles share the same edge points. This is the constraint at work.

In [ ]:
SEED = None
NX, NY = 24, 12
rng = new_rng(SEED)

tiles = [parse_tile(TILE_TEXT[k]) for k in REPS]

fig, ax = plt.subplots(figsize=(13, 6.8))
for y in range(NY):
    for x in range(NX):
        tile = rotate_tile(tiles[rng.integers(len(tiles))], rng.integers(4))
        for a, b in tile:
            c = tile_curve(a, b)
            ax.plot(c[:, 0] + x, c[:, 1] + y, color='#1d3557', lw=2, solid_capstyle='round')
ax.set_title('Random representative tile + random rotation in every cell: curves always meet at edge midpoints')
ax.set_aspect('equal'); ax.axis('off')
plt.tight_layout(); plt.show()


**Write your own tile.** Add a string to `TILE_TEXT` pairing the 8 points however you like, for example `"0-3 1-2 4-7 5-6"`. It works with everything below unchanged. Pairs that cross (such as `"0-4 1-5 ..."`) give over/under tiles, but thick strands will overlap where they cross.

---
## 5. The final weave: choose a tile by index

**Principle.** Now combine everything: **a tile from the library** (chosen by index), **rotations from a smooth field**, **colors from a second smooth field**, and **parallel strands**.

**Recipe**
1. `TILE_IDS = [7]` picks one tile for the whole picture. With several ids, for example `[7, 3, 12]`, a third smooth field decides which tile goes where.
2. `ROTATIONS = [0, 1]` lists the allowed quarter-turns: two states like the classic tile, or `[0, 1, 2, 3]` for all four.
3. A smooth field is split into equally common bins to choose rotation (and tile). `to_index` does this with quantiles.
4. Each curve is drawn as `N_STRANDS` parallel strands, colored by the color field plus jitter.

**Look at the randomness first.** The first cell plots the three per-tile maps (rotation, color, which tile) *before* any drawing, so you can see exactly which decisions the artwork is made from. The second cell draws the artwork.

In [ ]:
# =====================================================================
# SETTINGS
# =====================================================================
TILE_IDS      = [3]          # representative tile(s): [3] = one tile everywhere,
                             # [3, 6, 9] = mix representatives by a smooth field
ROTATIONS     = [0, 1]       # allowed quarter-turns: [0,1] or [0,1,2,3]
SEED          = 7            # None = new result every run; a number = repeatable
NX, NY        = 16, 20
BG            = '#f1e9d2'
ORIENT_SMOOTH = 1.2          # size of the same-rotation regions
COLOR_SMOOTH  = 3.0          # size of the color regions
N_STRANDS     = 3            # parallel strands per curve
BAND          = 0.055        # spacing between strands (tile = 1.0)
STRAND_SHIFT  = 0.09         # how far strands drift along the palette
JITTER        = 0.04         # small independent color variation per curve

rng = new_rng(SEED)
palette = make_cmap(['#d9674a', '#e8a56b', '#e6d28f', '#9db08c', '#5f8a86', '#2c3a35'])

# line width in points that just fills the strand spacing (the figure is 8 inches wide)
STRAND_W = BAND * (8 * 72 / NX) * 0.9

# ---------- THREE INDEPENDENT SMOOTH FIELDS, ONE VALUE PER TILE ----------
orient_field = smooth_field((NY, NX), ORIENT_SMOOTH, rng)         # decides rotation
color_field  = smooth_field((NY, NX), COLOR_SMOOTH, rng)          # decides color
pick_field   = smooth_field((NY, NX), ORIENT_SMOOTH * 2, rng)     # decides which tile (if several)

rot_idx  = to_index(orient_field, len(ROTATIONS))
tile_idx = to_index(pick_field, len(TILE_IDS))
tiles    = [parse_tile(TILE_TEXT[k]) for k in TILE_IDS]
offsets  = np.linspace(-1, 1, N_STRANDS) if N_STRANDS > 1 else np.array([0.0])

# ---------- PLOT THE RANDOMNESS FIRST (one value per tile) ----------
fig_r, axs = plt.subplots(1, 3, figsize=(12, 4.8))
axs[0].imshow(rot_idx, origin='lower', cmap='Greys', interpolation='nearest')
axs[0].set_title('Rotation per tile (from the smooth orientation field)')
axs[1].imshow(squash(color_field), origin='lower', cmap=palette, vmin=0, vmax=1, interpolation='nearest')
axs[1].set_title('Color value per tile (from the smooth color field)')
axs[2].imshow(tile_idx, origin='lower', cmap='tab10', vmin=0, vmax=9, interpolation='nearest')
axs[2].set_title('Which tile per tile (only varies if TILE_IDS has several)')
for a in axs: a.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# ---------- BUILD EVERY CURVE ----------
segments, colors = [], []
for j in range(NY):
    for i in range(NX):
        tile = rotate_tile(tiles[tile_idx[j, i]], ROTATIONS[rot_idx[j, i]])
        for a, b in tile:
            base = tile_curve(a, b) + [i, j]                               # move tile to its grid cell
            t = squash(color_field[j, i]) + rng.normal(0, JITTER)          # the coloring recipe
            for off in offsets:
                segments.append(offset(base, off * BAND * (N_STRANDS - 1) / 2))
                colors.append(palette(np.clip(t + off * STRAND_SHIFT, 0, 1)))

# ---------- DRAW ----------
fig = plt.figure(figsize=(8, 10), dpi=300)
ax = fig.add_axes([0, 0, 1, 1])
fig.patch.set_facecolor(BG)
ax.set_xlim(0, NX); ax.set_ylim(0, NY); ax.axis('off')
ax.add_collection(LineCollection(segments, colors=colors, linewidths=STRAND_W,
                                 capstyle='butt', antialiased=True))
finish(fig_to_img(fig), 'truchet_weave.png', grain=3)


---
## Cheat sheet

| Idea | Where | Takeaway |
|------|-------|----------|
| Edge midpoints shared | Step 1 | Any orientation of any tile joins its neighbours |
| Smoothing a coin-flip field | Step 2 | `sigma` controls the size of same-orientation regions |
| Separate field for color | Step 3 | Pattern and color regions interfere, which gives depth |
| Palindromic, identical edges | Step 4 | The only constraint; the tile interior is free |
| Tile as text | Step 4, 5 | Easy to index, mix and invent tiles |
| Rotation = shift labels by 2 | Step 4 | Four orientations without redrawing |

**Experiments that teach the most**
- Run step 2 with `SIGMAS = [0, 6, 12]` and see how the regions grow.
- In step 5, try every `TILE_ID` from the representative list [0, 1, 3, 4, 6, 9] with `ROTATIONS = [0, 1]`, then with `[0, 1, 2, 3]`.
- Mix tiles with `TILE_IDS = [3, 6, 9]` and watch the third map in the randomness plot.
- Write your own tile string and add it to `TILE_TEXT`.
- Swap in your own palette (5 to 7 hex colors, one dark and one light).
- Use the same smooth field for color and rotation to see the rigid, aligned look.

**If it looks wrong**
- *Strands collide in tight curves:* lower `BAND` or `N_STRANDS`.
- *Looks like static:* raise `ORIENT_SMOOTH`.
- *Looks flat:* add a second tile (`TILE_IDS`), use all four `ROTATIONS`, or widen the palette.

## Further reading

- Conceptual background: [Patterns built from tiles](conceptualGuide.md#4-patterns-built-from-tiles) in `conceptualGuide.md`.
- Implementation details: [Truchet weave](recipes.md#5-truchet-weave) in `recipes.md`.